In [1]:
import os
import requests
import pandas as pd

# Web-API (WFS) der Stadt Zürich: alle öffentlich zugänglichen Parkhäuser als GeoJSON
url = ("https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Oeffentlich_zugaengliche_Parkhaeuser"
       "?SERVICE=WFS&VERSION=1.1.0&REQUEST=GetFeature&TYPENAME=poi_parkhaus_view"
       "&OUTPUTFORMAT=application/vnd.geo%2Bjson&SRSNAME=EPSG:4326")
geo = requests.get(url, timeout=60).json()

# Jedes "feature" ist ein Parkhaus: Eigenschaften + Punkt-Koordinaten
rows = []
for feature in geo["features"]:
    p = feature["properties"]
    lon, lat = feature["geometry"]["coordinates"]
    rows.append({
        "city_name": p["name"].strip(),
        "city_address": p["adresse"],
        "capacity": p["anzahl_oeffentliche_pp"],
        "category": p["kategorie"],
        "latitude": lat,
        "longitude": lon,
    })

city = pd.DataFrame(rows)

# Rohdaten speichern (eigener Ordner, damit der Roboter nicht reinkommt)
os.makedirs("../data/external", exist_ok=True)
city.to_csv("../data/external/city_garages.csv", index=False)

print(city.shape)
city.head(10)

(136, 6)


,city_name,city_address,capacity,category,latitude,longitude
0,A-Park,Albisriederstrasse 348,74.0,Parkhaus,47.375637,8.486451
1,Accu,Fritz-Heeb-Weg 5,213.0,Parkhaus,47.414853,8.540591
2,Airgate,Leutschenbachstrasse 37,163.0,Parkhaus,47.415035,8.552979
3,Albisriederdörfli,Bachwiesenstrasse 141,26.0,Parkhaus,47.376036,8.485043
4,Albisriederplatz,Badenerstrasse 380c,65.0,Parkhaus,47.379393,8.508120
5,Aldi,Ueberlandstrasse 441,58.0,Parkhaus,47.405736,8.592106
6,Aldi Seebach,Schärenmoosstrasse 77a,70.0,Parkhaus,47.419748,8.556535
7,Altstetten-Luggweg,Luggwegstrasse 107a,44.0,Parkhaus,47.385053,8.490689
8,Atrium 565,Badenerstrasse 565b,18.0,Parkhaus,47.383777,8.497597
9,Bellevue-Garage,Stadelhoferstrasse 42,20.0,Parkhaus,47.367742,8.546466


In [2]:
import glob
import re

# 1) Alle Parkhäuser aus dem Feed: jede parking_id einmal (aus allen Monats-CSVs)
feed_files = sorted(glob.glob("../data/raw/parking_*.csv"))
feed_raw = pd.concat([pd.read_csv(f) for f in feed_files], ignore_index=True)
feed = (feed_raw[["parking_id", "parking_name", "address"]]
        .drop_duplicates(subset="parking_id", keep="last")
        .reset_index(drop=True))
print("Feed-Parkhäuser:", len(feed))

# 2) Namen vereinheitlichen, damit unterschiedliche Schreibweisen zusammenpassen
def normalize(name):
    name = re.sub(r"^(Parkhaus|Parkplatz)\s+", "", str(name).strip())
    name = name.lower()
    for a, b in [("ä", "ae"), ("ö", "oe"), ("ü", "ue"), ("é", "e")]:
        name = name.replace(a, b)
    return re.sub(r"[^a-z0-9]", "", name)

feed["key"] = feed["parking_name"].apply(normalize)
city["key"] = city["city_name"].apply(normalize)

# 3) Abgleich: alle Feed-Parkhäuser behalten (left join)
matched = feed.merge(city, on="key", how="left")

found = matched["capacity"].notna().sum()
print(f"Automatisch zugeordnet: {found} von {len(feed)}")
print("\nNicht zugeordnet:")
matched.loc[matched["capacity"].isna(), ["parking_id", "parking_name", "address"]]

Feed-Parkhäuser: 37
Automatisch zugeordnet: 29 von 37

Nicht zugeordnet:


,parking_id,parking_name,address
10,hafen_enge,Parkhaus Hafen Enge,Mythenquai 21
18,messe,Parkhaus Messe Zürich AG,Andreasstrasse 65
28,unispital_nord,Parkhaus USZ Nord,Frauenklinikstrasse
29,uni_irchel,Parkhaus Uni Irchel,Winterthurerstrasse 181
32,zueri11,Parkhaus Züri 11 Shopping,Nansenstrasse 5/7
34,theater_11,Parkplatz Theater 11,Dörfli-/Thurgauerstrasse
35,unispital_sued,Parkplatz USZ Süd,Gloriastrasse
36,puls5,Puls 5 Parkgarage,Giessereistrasse 18


In [5]:
import time

# 4) Manuelle Zuordnung: gleiche Parkhäuser, aber andere Namen im Stadt-Datensatz
manual = {
    "messe": "Messe",
    "puls5": "Puls 5",
    "uni_irchel": "Universität Irchel",
    "unispital_nord": "Unispital Nord",
    "zueri11": "Züri 11",
}
city_by_name = city.set_index("city_name", drop=False)
matched["match_method"] = matched["capacity"].notna().map({True: "name", False: None})

for pid, city_name in manual.items():
    i = matched.index[matched["parking_id"] == pid][0]
    for col in ["city_name", "city_address", "capacity", "category", "latitude", "longitude"]:
        matched.loc[i, col] = city_by_name.loc[city_name, col]
    matched.loc[i, "match_method"] = "manual"

# 5) Web Scraping: Kapazität von der Detailseite auf pls-zh.ch für den Rest
from bs4 import BeautifulSoup

def scrape_capacity(pid):
    url = f"https://www.pls-zh.ch/parkhaus/{pid}.jsp?pid={pid}"
    r = requests.get(url, headers={"User-Agent": "ZHAW-student-project"}, timeout=30)
    text = BeautifulSoup(r.text, "html.parser").get_text(" ", strip=True)  # HTML -> reiner Text
    m = re.search(r"Angebot:\s*(\d+)\s*Pl", text)
    return int(m.group(1)) if m else None

for i in matched.index[matched["capacity"].isna()]:
    pid = matched.loc[i, "parking_id"]
    cap = scrape_capacity(pid)
    time.sleep(1)  # höflich: nicht zu schnell hintereinander anfragen
    if cap:
        matched.loc[i, "capacity"] = cap
        matched.loc[i, "match_method"] = "scraped"
    else:
        matched.loc[i, "match_method"] = "none"

# 6) Übersicht für die Dokumentation
print(matched["match_method"].value_counts())

# 7) Garagen-Tabelle speichern (für A und C)
garages = matched[["parking_id", "parking_name", "capacity", "latitude", "longitude",
                   "city_name", "city_address", "match_method"]]
os.makedirs("../data/processed", exist_ok=True)
garages.to_csv("../data/processed/garages.csv", index=False)
garages

match_method
name       29
manual      5
scraped     2
none        1
Name: count, dtype: int64


,parking_id,parking_name,capacity,latitude,longitude,city_name,city_address,match_method
0,accu,Parkhaus Accu,213.0,47.414853,8.540591,Accu,Fritz-Heeb-Weg 5,name
1,albisriederplatz,Parkhaus Albisriederplatz,65.0,47.379393,8.508120,Albisriederplatz,Badenerstrasse 380c,name
2,bleicherweg,Parkhaus Bleicherweg,258.0,47.367372,8.535696,Bleicherweg,Beethovenstrasse 35,name
3,center_11,Parkhaus Center Eleven,377.0,47.412826,8.540275,Center Eleven,Sophie-Taeuber-Strasse 4,name
4,cp,Parkhaus City Parking,607.0,47.375690,8.535496,City Parking,Gessnerallee 14,name
5,cityport,Parkhaus Cityport,131.0,47.410875,8.540895,Cityport,Affolternstrasse 56,name
6,crowne_plaza,Parkhaus Crowne Plaza,310.0,47.381153,8.506867,Crowne Plaza,Bienenstrasse 9,name
7,dorflinde,Parkhaus Dorflinde,48.0,47.407595,8.548751,Dorflinde,Schwamendingenstrasse 31,name
8,feldegg,Parkhaus Feldegg,147.0,47.360383,8.553527,Feldegg,Riesbachstrasse 6,name
9,globus,Parkhaus Globus,139.0,47.376055,8.537717,Globus,Löwenstrasse 50a,name


In [8]:
# ===== Schritt 5: Bereinigung =====
raw = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("../data/raw/parking_*.csv"))], ignore_index=True)
garages = pd.read_csv("../data/processed/garages.csv").drop(columns=["max_free", "capacity_used"], errors="ignore")
raw["fetched_at_utc"] = pd.to_datetime(raw["fetched_at_utc"], utc=True)
raw["source_updated_at_utc"] = pd.to_datetime(raw["source_updated_at_utc"], utc=True)

log = [("Rohdaten", len(raw))]  # Protokoll: Zeilen nach jeder Regel
df = raw.copy()

# Regel 1: Zeilen ohne Messwert
df = df[(df["status"] != "unknown") & df["free_spaces"].notna()]
log.append(("1: ohne Messwert entfernt", len(df)))

# Regel 2: Eingefrorene Werte (seit über 48 h keine Änderung gemeldet)
df["lag_min"] = (df["fetched_at_utc"] - df["source_updated_at_utc"]).dt.total_seconds() / 60
df = df[df["lag_min"] <= 48 * 60]
log.append(("2: eingefrorene Werte (> 48 h) entfernt", len(df)))

# Regel 3: Hafen Enge (keine Kapazität) und Messe (Anzeige endet bei 999)
df = df[~df["parking_id"].isin(["hafen_enge", "messe"])]
log.append(("3: Hafen Enge und Messe entfernt", len(df)))

# Regel 4: Duplikate im selben 15-Minuten-Fenster
df["slot_utc"] = df["fetched_at_utc"].dt.floor("15min")
df = df.sort_values("fetched_at_utc").drop_duplicates(subset=["slot_utc", "parking_id"], keep="first")
log.append(("4: Duplikate pro 15-Min.-Fenster entfernt", len(df)))

# Regel 5: Kapazität nie kleiner als die höchste gemeldete Zahl freier Plätze
max_free = df.groupby("parking_id")["free_spaces"].max().rename("max_free")
garages = garages.merge(max_free, on="parking_id", how="left")
garages["capacity_used"] = garages[["capacity", "max_free"]].max(axis=1)
adjusted = garages[garages["max_free"] > garages["capacity"]]
print("Kapazität angepasst bei:", ", ".join(adjusted["parking_id"]))

# Auslastung und Zeitmerkmale (Lokalzeit Zürich)
df = df.merge(garages[["parking_id", "capacity_used"]], on="parking_id", how="left")
df["occupancy_rate"] = (df["capacity_used"] - df["free_spaces"]) / df["capacity_used"]
local = df["slot_utc"].dt.tz_convert("Europe/Zurich")
df["date"] = local.dt.date
df["hour"] = local.dt.hour
df["weekday"] = local.dt.day_name()
df["is_weekend"] = local.dt.day_of_week >= 5

# Protokoll anzeigen
log = pd.DataFrame(log, columns=["Schritt", "Zeilen"])
log["entfernt"] = -log["Zeilen"].diff().fillna(0).astype(int)
print(log.to_string(index=False))

# Speichern
clean = df[["slot_utc", "parking_id", "status", "free_spaces", "capacity_used", "occupancy_rate",
            "date", "hour", "weekday", "is_weekend"]]
clean.to_csv("../data/processed/occupancy_clean.csv", index=False)
garages.to_csv("../data/processed/garages.csv", index=False)
print("\nSaubere Daten:", clean.shape, "| Parkhäuser:", clean["parking_id"].nunique())

Kapazität angepasst bei: cityport, helvetia, jungholz, octavo, stampfenbach, urania, puls5
                                  Schritt  Zeilen  entfernt
                                 Rohdaten   68006         0
                1: ohne Messwert entfernt   65320      2686
  2: eingefrorene Werte (> 48 h) entfernt   63027      2293
         3: Hafen Enge und Messe entfernt   60646      2381
4: Duplikate pro 15-Min.-Fenster entfernt   57742      2904

Saubere Daten: (57742, 10) | Parkhäuser: 33
